# 실제자료의 측정·표본 사양 탐색 — preliminary 전용
이 노트북은 본 연구에서 사용하지 않는 별도 탐색이다. 이미 이전 결과를 본 상태에서 공간·가용률·노출·결과 정의를 바꾸어 본다. **가장 작은 p값도 확인적 발견으로 해석하지 않는다.** 실제자료 탐색만 수행하며 자료 위조·결측 0 대체·의도적 오용·효과 주입 모의실험은 이 노트북에 포함하지 않는다.

전체 격자는 2개 공간×4개 가용률×2개 노출×3개 결과=48개 결합 회귀 시스템, 업종별 480개 계수 검정이다. 중심점 500m·가용률 90%·예정회차에 대한 야간 단위 민감도 4시스템·40개를 추가한다. **총 52시스템·520개 시도**를 기록하며 원단위 야간 10개는 격자와 중복이라는 표시를 남긴다. 결과를 보고 후보를 늘리거나 유의한 결과만 저장하지 않는다.

## 1. 고정 입력과 보호 범위
`jupyter-notebook/sandbox/metadata/lab_plan.json`의 보호 파일, 12번 완료 명세와 패널 해시를 확인한다. 분석 함수는 검증된 05·13번 노트북에서 함수 정의만 AST로 읽어 재사용한다. 원본과 final 산출물은 수정하지 않는다.

In [ ]:
# 현재 구조 및 과거 고정 명세의 경로를 함께 해석합니다.
from pathlib import Path as _ProjectPath
import json as _bootstrap_json
_project_home = next(p for p in [_ProjectPath.cwd(), *_ProjectPath.cwd().parents]
                     if (p / 'environment/project_paths.ipynb').is_file())
_project_notebook = _bootstrap_json.loads((_project_home / 'environment/project_paths.ipynb').read_text(encoding='utf-8'))
for _project_cell in _project_notebook['cells']:
    if 'project-runtime' in _project_cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(_project_cell['source']), 'environment/project_paths.ipynb', 'exec'), globals())

from pathlib import Path
import os
for key in ['OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','OMP_NUM_THREADS']:os.environ[key]='1'
import ast,json,hashlib,platform
from datetime import datetime,timezone
import numpy as np
import pandas as pd
from scipy import linalg,stats
from statsmodels.stats.multitest import multipletests
from IPython.display import display,Markdown
ROOT=PROJECT_ROOT
REPORTS=project_path('jupyter-notebook/sandbox/reports');DATA=project_path('jupyter-notebook/sandbox/data')
REPORTS.mkdir(parents=True,exist_ok=True);DATA.mkdir(parents=True,exist_ok=True)
sha=lambda p:hashlib.sha256(p.read_bytes()).hexdigest()
input_records={}
def register(relative,expected=None):
    actual=sha(project_path(relative))
    if expected is not None:assert actual==expected,relative
    input_records[relative]={'path':relative,'sha256_before':actual,'expected_sha256':expected}
    return project_path(relative)
lab=project_json_loads(register('jupyter-notebook/sandbox/metadata/lab_plan.json').read_text(encoding='utf-8'))
assert lab['exploratory'] and not lab['confirmatory']
for relative,digest in lab['protected_sha256'].items():register(relative,digest)
s12=project_json_loads(register('reports/20261006/revised_quarterly/12_dataset_summary.json').read_text(encoding='utf-8'))
assert s12['status']=='complete' and s12['all_inputs_unchanged'] and s12['validation_all_passed']
PLAN=project_json_loads(register(s12['plan_path'],s12['plan_sha256']).read_text(encoding='utf-8'))
KEY=['area_id','industry_id','quarter'];INDUSTRY_IDS=[f'CS100{i:03d}' for i in range(1,11)]
panel=pd.read_csv(register(s12['panel_path'],s12['panel_sha256']),dtype={k:str for k in KEY},float_precision='round_trip')
assert len(panel)==18120 and not panel.duplicated(KEY).any()
INDUSTRIES=dict(panel[['industry_id','industry']].drop_duplicates().sort_values('industry_id').to_numpy())
assert list(INDUSTRIES)==INDUSTRY_IDS
SOURCE05='jupyter-notebook/20261006/notebooks/05_외식업_고정효과분석.ipynb';SOURCE13='jupyter-notebook/20261006/notebooks/13_분기매출액_수정분석.ipynb'
register(SOURCE05);register(SOURCE13)

In [ ]:
def import_functions(relative,names):
    notebook=project_json_loads((project_path(relative)).read_text(encoding='utf-8'));found=[]
    for cell in notebook['cells']:
        if cell['cell_type']!='code':continue
        source=''.join(cell['source'])
        if any(line.lstrip().startswith(('%','!')) for line in source.splitlines()):continue
        found.extend(node for node in ast.parse(source).body if isinstance(node,ast.FunctionDef) and node.name in names)
    assert sorted(node.name for node in found)==sorted(names)
    exec(compile(ast.fix_missing_locations(ast.Module(body=found,type_ignores=[])),relative,'exec'),globals())
    return {node.name:hashlib.sha256(ast.dump(node).encode()).hexdigest() for node in found}
function_hashes={**import_functions(SOURCE05,['absorb_effects']),
    **import_functions(SOURCE13,['key_hash','fit_system','estimate_row'])}
assert PLAN['minimum_industry_clusters']==20 and PLAN['minimum_observations_per_area_industry']==2
common=panel.total_sales.gt(0)&panel.night_sales.notna()&panel.night_sales.ge(0)
assert panel.loc[common,'night_sales'].le(panel.loc[common,'total_sales']).all()
panel['log1p_total_won']=np.log1p(panel.total_sales.where(panel.total_sales.gt(0)))
panel['log1p_night_won']=np.log1p(panel.night_sales.where(panel.night_sales.ge(0)))
panel['night_share_pct']=100*panel.night_sales/panel.total_sales.where(panel.total_sales.gt(0))
panel['log1p_night_thousand_won']=np.log1p(panel.night_sales.where(panel.night_sales.ge(0))/1000)
panel['log1p_night_million_won']=np.log1p(panel.night_sales.where(panel.night_sales.ge(0))/1000000)
panel['log_positive_night_won']=np.log(panel.night_sales.where(panel.night_sales.gt(0)))
assert int(common.sum())==7661
print('고정 원패널 18,120행, 공통 결과 가용 7,661행')

## 2. 결과를 보기 전에 유한한 격자 명시
공간은 중심점 500m/폴리곤 최단거리 500m, ID 가용률은 100/95/90/80%, 노출은 알려진 예정 시작회차/후보 공연 ID 수다. ID 노출에도 같은 가용률 기준을 적용한다. 기본 격자는 세 결과에 동일한 원 관측표본을 사용한다.

100% 미만의 예정회차는 **미가용 후보를 뺀 부분 합계**다. ID 가용률이 회차·관객의 관측률은 아니다. 긍정 야간 로그는 0원을 빼는 다른 표본이므로 화폐 단위 변경과 별개로 표시한다.

In [ ]:
SPACES=['centroid500','min_distance500'];THRESHOLDS=[100,95,90,80]
EXPOSURES={'scheduled':'known_scheduled_start_count','id_count':'candidate_id_count'}
OUTCOMES={'total_won':'log1p_total_won','night_won':'log1p_night_won','night_share':'night_share_pct'}
registry_rows=[]
for space in SPACES:
    for threshold in THRESHOLDS:
        for exposure,exposure_column in EXPOSURES.items():
            group=f'{space}_a{threshold:03d}_{exposure}'
            for outcome,column in OUTCOMES.items():
                registry_rows.append(dict(spec_id=f'grid_{group}_{outcome}',family='measurement_sample_grid',
                    family30=group,spatial_definition=space,availability_threshold_pct=threshold,
                    exposure=exposure,exposure_column=exposure_column,outcome=column,
                    positive_night_only=False,duplicate_of=None))
for unit,column in [('won','log1p_night_won'),('thousand_won','log1p_night_thousand_won'),
    ('million_won','log1p_night_million_won'),('positive_only_log_won','log_positive_night_won')]:
    registry_rows.append(dict(spec_id='currency_'+unit,family='night_currency_sensitivity',family30=None,
        spatial_definition='centroid500',availability_threshold_pct=90,exposure='scheduled',
        exposure_column='known_scheduled_start_count',outcome=column,
        positive_night_only=unit=='positive_only_log_won',
        duplicate_of='grid_centroid500_a090_scheduled_night_won' if unit=='won' else None))
registry=pd.DataFrame(registry_rows)
assert len(registry)==52 and registry.spec_id.nunique()==52
assert registry.family.eq('measurement_sample_grid').sum()==48
registry.to_csv(DATA/'01_specification_registry.csv',index=False,encoding='utf-8-sig')
display(registry.groupby(['family','spatial_definition','availability_threshold_pct'],dropna=False).size().rename('회귀시스템수').reset_index())

## 3. 고정효과·상권 군집 추론과 모든 실패 기록
13번과 같은 상권×업종 및 업종×분기 고정효과를 사용한다. 상권×업종에 2분기 이상, 업종별 최소 20개 상권을 요구한다. 업종별 기울기를 결합한 모형이므로 CR1 보정과 t 자유도는 **시스템 전체 상권 수 G−1**이다. 업종별 상권 수와 G를 별도 저장한다.

미추정 업종과 실행 실패도 예정된 10개 행으로 남긴다. 일반적인 p값과 개별 구간은 각 사양을 고정했을 때의 조건부 수치이며 사양 탐색을 거친 발견의 검증력을 뜻하지 않는다.

In [ ]:
def make_frame(spec):
    f=panel.loc[common & panel[f'availability_pct_{spec.spatial_definition}'].ge(spec.availability_threshold_pct)].copy()
    n_before_positive=len(f)
    if spec.positive_night_only:f=f.loc[f.night_sales.gt(0)].copy()
    f=f.loc[f[spec.outcome].notna()].copy()
    counts=f.groupby(['area_id','industry_id']).quarter.transform('nunique')
    f=f.loc[counts.ge(2)].sort_values(KEY).reset_index(drop=True)
    f['exposure_count']=f[f'{spec.exposure_column}_{spec.spatial_definition}']
    assert f.exposure_count.notna().all() and f.exposure_count.ge(0).all()
    f['x']=np.log1p(f.exposure_count)
    return f,n_before_positive

result_rows=[];model_rows=[];sample_frames={};models_for_validation={}
for number,spec in enumerate(registry.itertuples(index=False),1):
    frame,n_before_positive=make_frame(spec);sample_frames[spec.spec_id]=frame
    failure=None
    try:m=fit_system(frame,spec.outcome)
    except Exception as exc:
        failure=f'{type(exc).__name__}: {str(exc)[:500]}'
        m={'status':'model_failed','blocks':[],'eligibility':[], 'nobs':0,'clusters':0,'rank':0}
    position={block['industry_id']:j for j,block in enumerate(m['blocks'])}
    model_rows.append(dict(spec_id=spec.spec_id,family=spec.family,outcome=spec.outcome,model_status=m['status'],
        failure_reason=failure,n_before_positive_filter=n_before_positive,n_common_before_industry_filter=len(frame),
        nobs_model=m['nobs'],clusters_model=m['clusters'],inference_df=m.get('df',np.nan),design_rank=m['rank'],
        cluster_correction=m.get('correction',np.nan),estimated_industries=len(m['blocks']),
        common_sample_key_sha256=key_hash(frame),night_zero_rows=int(frame.night_sales.eq(0).sum()),
        positive_night_only=spec.positive_night_only,duplicate_of=spec.duplicate_of))
    for ind in INDUSTRY_IDS:
        part=frame.loc[frame.industry_id.eq(ind)]
        eligibility=next((e for e in m['eligibility'] if e['industry_id']==ind),{})
        row={**spec._asdict(),'industry_id':ind,'industry':INDUSTRIES[ind],
            'status':eligibility.get('status','model_failed' if failure else 'not_estimable'),
            'failure_reason':failure,'nobs_industry':len(part),'clusters_industry':int(part.area_id.nunique()),
            'nobs_model':m['nobs'],'clusters_model':m['clusters'],'inference_df':m.get('df',np.nan),
            'cluster_correction':m.get('correction',np.nan),'design_rank':m['rank'],
            'residual_x_sd':eligibility.get('residual_x_sd',np.nan),
            'sample_key_sha256':key_hash(part),'coefficient':np.nan,'standard_error':np.nan,
            'ci_low':np.nan,'ci_high':np.nan,'p_value':np.nan}
        if ind in position:row.update(estimate_row(m,position[ind]))
        result_rows.append(row)
    if spec.spec_id in ['grid_centroid500_a100_scheduled_total_won','grid_centroid500_a100_scheduled_night_won']:
        models_for_validation[spec.spec_id]=m
    if number%8==0 or number==len(registry):print(f'고정 탐색 진행 {number}/{len(registry)}시스템')
results=pd.DataFrame(result_rows);model_samples=pd.DataFrame(model_rows)
assert len(results)==520 and results.groupby('spec_id').size().eq(10).all()
assert not results.duplicated(['spec_id','industry_id']).any()
display(model_samples.groupby('family').agg(systems=('spec_id','size'),failed=('model_status',lambda x:x.eq('model_failed').sum()),estimated_coefficients=('estimated_industries','sum')))

## 4. 표본 일치와 기존 결과 재현 점검
격자 안의 ID/예정회차 비교는 같은 공간·가용률·결과에 대해 동일한 후보 관측행을 사용한다. 업종별 식별 실패가 있으면 적합 표본은 달라질 수 있으므로 미추정 행을 숨기지 않는다. 원단위 야간의 중복 시스템도 결과가 같은지 검증한다.

In [ ]:
checks=[]
for space in SPACES:
    for threshold in THRESHOLDS:
        for outcome in OUTCOMES:
            left=f'grid_{space}_a{threshold:03d}_scheduled_{outcome}'
            right=f'grid_{space}_a{threshold:03d}_id_count_{outcome}'
            same=key_hash(sample_frames[left])==key_hash(sample_frames[right])
            assert same;checks.append(dict(check='fair_candidate_sample:'+left,passed=same,max_error=0))
prior=pd.read_csv(register('reports/20261006/revised_quarterly/13_effects.csv'),float_precision='round_trip')
for suffix,old_outcome in [('total_won','log1p_total_sales_won'),('night_won','log1p_night_sales_21_24_won')]:
    new=results.loc[results.spec_id.eq('grid_centroid500_a100_scheduled_'+suffix)].set_index('industry_id')
    old=prior.loc[prior.scenario.eq('primary_centroid100')&prior.outcome.eq(old_outcome)].set_index('industry_id')
    assert new.status.eq(old.status).all()
    error=float(np.max(np.abs(new[['coefficient','standard_error','ci_low','ci_high','p_value']].to_numpy()-old[['coefficient','standard_error','ci_low','ci_high','p_value']].to_numpy())))
    assert error<1e-8
    checks.append(dict(check='reproduce_final13:'+suffix,passed=True,max_error=error))
left=results.loc[results.spec_id.eq('currency_won')].set_index('industry_id')
right=results.loc[results.spec_id.eq('grid_centroid500_a090_scheduled_night_won')].set_index('industry_id')
assert left.sample_key_sha256.eq(right.sample_key_sha256).all()
assert np.allclose(left[['coefficient','standard_error','p_value']],right[['coefficient','standard_error','p_value']],equal_nan=True,rtol=0,atol=1e-12)
checks.append(dict(check='currency_won_duplicate_exact',passed=True,max_error=0))
validation=pd.DataFrame(checks);display(validation.tail(3))

## 5. 탐색군과 다중비교 분모 공개
BH10은 각 시스템의 10개 업종, BH30은 격자의 공간×가용률×노출에 속한 세 결과 30개다. 별도로 격자 480개, 단위 민감도 40개, 전체 시도 520개의 BH를 제공한다. 미추정·실패는 계산에 p=1로 포함하되 실제 추정된 결과처럼 표시하지 않는다.

전체 520개에는 중복된 10개가 포함된다. 중복 제거한 독립 검정 수가 510개라는 뜻도 아니다. 사양 간 자료와 표본을 공유하며 탐색 이후 정한 이 다중비교 보정 자체가 확인적 설계를 만들어주지는 않는다.

In [ ]:
results['p_for_multiplicity']=results.p_value.where(results.status.eq('estimated'),1).fillna(1)
results['p_bh10']=np.nan;results['p_bh30']=np.nan;results['p_bh_family']=np.nan
for _,indices in results.groupby('spec_id').groups.items():
    assert len(indices)==10
    results.loc[indices,'p_bh10']=multipletests(results.loc[indices,'p_for_multiplicity'],method='fdr_bh')[1]
for _,indices in results.loc[results.family.eq('measurement_sample_grid')].groupby('family30').groups.items():
    assert len(indices)==30
    results.loc[indices,'p_bh30']=multipletests(results.loc[indices,'p_for_multiplicity'],method='fdr_bh')[1]
for family,size in [('measurement_sample_grid',480),('night_currency_sensitivity',40)]:
    indices=results.index[results.family.eq(family)];assert len(indices)==size
    results.loc[indices,'p_bh_family']=multipletests(results.loc[indices,'p_for_multiplicity'],method='fdr_bh')[1]
results['p_bh_all520']=multipletests(results.p_for_multiplicity,method='fdr_bh')[1]
results['multiplicity_family_size']=np.where(results.family.eq('measurement_sample_grid'),480,40)
results['pooled_attempt_family_size']=520
results['adjusted_values_are_only_for_estimated_rows']=results.status.eq('estimated')
for source,target in [('coefficient','contrast_double_plus1'),('ci_low','contrast_ci_low'),('ci_high','contrast_ci_high')]:
    results[target]=np.where(results.outcome.eq('night_share_pct'),results[source]*np.log(2),100*np.expm1(results[source]*np.log(2)))
results['contrast_unit']=np.where(results.outcome.eq('night_share_pct'),'percentage_point_night_share',
    np.where(results.outcome.eq('log_positive_night_won'),'percent_geometric_mean_positive_night_won','percent_geometric_mean_one_plus_sales_in_selected_currency_unit'))
results['exploratory_not_confirmatory']=True
counts=results.groupby(['family','spec_id','outcome']).agg(planned_tests=('industry_id','size'),
    estimated=('status',lambda x:x.eq('estimated').sum()),raw_p_below05=('p_value',lambda x:x.lt(.05).sum()),
    bh10_below05=('p_bh10',lambda x:x.lt(.05).sum()),bh30_below05=('p_bh30',lambda x:x.lt(.05).sum()),
    family_bh_below05=('p_bh_family',lambda x:x.lt(.05).sum()),all520_bh_below05=('p_bh_all520',lambda x:x.lt(.05).sum()),
    minimum_raw_p=('p_value','min')).reset_index()
display(counts)

**해석:** 야간 비중의 대비는 퍼센트포인트이며 총매출·야간 금액 로그의 상대차이와 단위가 다르다. 로그+1 금액의 대비는 선택한 화폐 단위로 표시한 `1+매출`의 조건부 기하평균 차이다. 서로 다른 화폐 단위의 결과를 같은 성장률 추정치로 비교하지 않는다.

## 6. 100%에서 제외됐던 핵심 상권의 복원
대학로(혜화역)와 홍대입구역(홍대)의 제공 매출행이 각 가용률에서 얼마나 남는지 표시한다. 이 표는 반복 관측 필터 후·업종별 최소 군집 필터 전의 표본 수다. 상권이 복원돼도 누락된 예정회차가 실제 관측된 것은 아니다.

In [ ]:
core_ids=['3120012','3120103'];core_rows=[]
for area_id in core_ids:
    full=panel.loc[common&panel.area_id.eq(area_id)]
    assert len(full)>0
    for space in SPACES:
        for threshold in THRESHOLDS:
            f=sample_frames[f'grid_{space}_a{threshold:03d}_scheduled_total_won']
            kept=f.loc[f.area_id.eq(area_id)]
            before_repeat=full.loc[full[f'availability_pct_{space}'].ge(threshold)]
            core_rows.append(dict(area_id=area_id,area_name=full.area_name.iloc[0],district=full.district.iloc[0],
                spatial_definition=space,availability_threshold_pct=threshold,
                original_provided_rows=len(full),rows_after_availability=len(before_repeat),rows_after_repeat=len(kept),
                quarters_after_repeat=int(kept.quarter.nunique()),industries_after_repeat=int(kept.industry_id.nunique()),
                retained_pct_of_provided=100*len(kept)/len(full),
                mean_ID_availability_pct=float(full[f'availability_pct_{space}'].mean()),
                sample_stage='before_minimum_industry_clusters'))
restored=pd.DataFrame(core_rows)
display(restored.loc[restored.spatial_definition.eq('centroid500') & restored.availability_threshold_pct.isin([100,90])])

## 7. 고정 중심점 90% 표본의 화폐 단위 민감도
원/천원/백만원 `log1p`는 야간 0을 보존한 같은 행이다. 양수만의 `log(원)`은 실제 야간 0을 제외하고 반복 관측 조건을 다시 적용하므로 표본이 다르다. 이 조건도 사전에 정한 40개 검정에 포함한다.

In [ ]:
currency=results.loc[results.family.eq('night_currency_sensitivity')].copy()
unit_samples=model_samples.loc[model_samples.family.eq('night_currency_sensitivity')]
zero_retaining=unit_samples.loc[~unit_samples.positive_night_only]
assert zero_retaining.common_sample_key_sha256.nunique()==1
positive_frame=sample_frames['currency_positive_only_log_won']
assert positive_frame.night_sales.gt(0).all()
display(unit_samples[['spec_id','n_common_before_industry_filter','nobs_model','clusters_model','night_zero_rows','positive_night_only']])
display(currency[['spec_id','industry','status','coefficient','standard_error','p_value','p_bh10','p_bh_family','p_bh_all520','nobs_industry','clusters_industry']])
family_counts=results.groupby('family').agg(attempts=('industry_id','size'),estimated=('status',lambda x:x.eq('estimated').sum()),
    raw_p_below05=('p_value',lambda x:x.lt(.05).sum()),family_bh_below05=('p_bh_family',lambda x:x.lt(.05).sum()),
    all520_bh_below05=('p_bh_all520',lambda x:x.lt(.05).sum()),minimum_raw_p=('p_value','min')).reset_index()
display(family_counts)
minimum=results.loc[results.p_value.idxmin()] if results.p_value.notna().any() else None
if minimum is not None:
    display(Markdown(f'**탐색에서 관측된 최소 p:** {minimum.p_value:.6g}; {minimum.spec_id}, {minimum.industry}. '
        '이는 전체 시도 중 선택된 수치이며 독립적으로 사전 지정한 확인적 결과가 아니다.'))

## 8. 전수 장부 저장과 보호 파일 재검증
모든 520개 예정 행, 52개 시스템 상태, 미추정·실패 이유, 핵심 상권 복원표와 검사 결과를 저장한다. 조정 p값에서 미추정 행에 붙은 값은 p=1 대입을 통한 탐색군 계산용이며 실제 추정치가 아니다. 본 연구 파일은 수정하지 않는다.

In [ ]:
output_frames={'01_specification_results.csv':results,'01_model_samples.csv':model_samples,
    '01_multiplicity_counts.csv':counts,'01_family_counts.csv':family_counts,
    '01_restored_core_areas.csv':restored,'01_validation.csv':validation}
output_records={}
for name,frame in output_frames.items():
    path=REPORTS/name;frame.to_csv(path,index=False,encoding='utf-8-sig')
    output_records[path.relative_to(ROOT).as_posix()]={'rows':len(frame),'columns':frame.columns.tolist(),'sha256':sha(path)}
registry_path=DATA/'01_specification_registry.csv'
output_records[registry_path.relative_to(ROOT).as_posix()]={'rows':len(registry),'columns':registry.columns.tolist(),'sha256':sha(registry_path)}
for relative,record in input_records.items():
    record['sha256_after']=sha(project_path(relative));record['unchanged']=record['sha256_after']==record['sha256_before']
assert all(row['unchanged'] for row in input_records.values())
def finite(value):
    if isinstance(value,dict):return {k:finite(v) for k,v in value.items()}
    if isinstance(value,list):return [finite(v) for v in value]
    if isinstance(value,(float,np.floating)) and not np.isfinite(value):return None
    if isinstance(value,np.generic):return value.item()
    return value
summary=finite({'status':'complete','created_utc':datetime.now(timezone.utc).isoformat(),
    'role':'actual_data_measurement_sample_exploration','exploratory':True,'confirmatory':False,
    'prior_results_seen':True,'all_inputs_unchanged':True,'network_requests':0,'raw_data_mutated':False,
    'planned_grid_systems':48,'planned_grid_tests':480,'planned_currency_systems':4,'planned_currency_tests':40,
    'attempted_systems':len(model_samples),'attempted_tests':len(results),'duplicate_coefficient_attempts':10,
    'distinct_specification_coefficient_combinations_excluding_exact_duplicate':510,
    'independent_test_count_known':False,'fitted_systems':int(model_samples.model_status.eq('estimated').sum()),
    'failed_systems':int(model_samples.model_status.eq('model_failed').sum()),
    'estimated_coefficients':int(results.status.eq('estimated').sum()),
    'unestimated_or_failed_coefficients':int(results.status.ne('estimated').sum()),
    'minimum_raw_p_row':minimum.to_dict() if minimum is not None else None,
    'raw_p_below05':int(results.p_value.lt(.05).sum()),
    'all520_bh_below05':int(results.p_bh_all520.lt(.05).sum()),
    'family_counts':family_counts.to_dict('records'),
    'currency_model_samples':unit_samples.to_dict('records'),
    'core_area_restoration':restored.to_dict('records'),
    'multiplicity':{'BH10':'each system10 industries','BH30':'grid space x availability x exposure,3 outcomes x10',
        'BH_family':'grid480 or currency40','BH_all520':'all attempted coefficients including10 intentional duplicates',
        'unestimated_p_for_calculation':1,'post_search_adjustment_is_not_confirmatory':True},
    'model_definition':{'fixed_effects':['area_industry','industry_quarter'],'cluster':'area_id',
        'inference':'joint system CR1, t(G_system-1); separate industry clusters retained',
        'minimum_industry_clusters':20,'minimum_entity_quarters':2,
        'common_grid_outcome_sample':'total_sales>0 and nonmissing night_sales>=0, no imputation',
        'currency_positive_only_changes_sample':True},
    'limitations':['partial scheduled sums below100pct do not reveal actual missing sessions or audience',
        'night share is a percentage point outcome and differs from night sales amount',
        'log1p monetary units change the estimand when zeros are present',
        'all specifications share data and are not independent replications',
        'minimum p is selected from an exploratory family, not a confirmatory discovery'],
    'function_AST_sha256':function_hashes,'validation':validation.to_dict('records'),
    'inputs':list(input_records.values()),'outputs':output_records,'python':platform.python_version()})
(REPORTS/'01_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
print(f'완료: {len(model_samples)}시스템·{len(results)}계수 시도 전수 기록; 입력 {len(input_records)}개 불변')

**최종 판단:** 사양을 여러 번 바꾸었을 때 발견되는 작은 p값과 보정 후 결과를 모두 공개한다. 유의한 항목이 있더라도 독립 자료나 사전 지정 후속 검증 없이 본 연구의 발견으로 옮기지 않는다. 유의한 결과가 없더라도 탐색을 더 늘려야 할 이유로 삼지 않는다.